# TM-Vec hits summary

Post-processing of the outputs produced by `run_tmvec_pipeline.sh`.

Steps:
1. **Check outputs** – flags which gene x proteome searches have finished.
2. **Filter hits by TM-score** – keeps only hits with TM-score >= `THRESHOLD`.
3. **Count table** – number of distinct query proteins with at least one hit, per proteome x gene.
4. **Pathway presence/absence** – assigns sialic acid–related functions to each genome based on the genes detected.

Set `THRESHOLD` in the cell below (this analysis was run with 0.70 and 0.90). All output files are named after the threshold.

In [7]:
import re
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------
THRESHOLD = 0.90                       # minimum TM-score to keep a hit
RAW_DIR = Path("/media/Kirk/thais/SialoCat/tmVEC/tmvec_DB6_outputs")    # raw outputs from run_tmvec_pipeline.sh
STATUS_CSV = "/media/Kirk/thais/SialoCat/tmVEC/tmvec_status.csv"        # table with one row per gene x proteome (columns: gene, proteome, status)

FILTERED_DIR = Path(f"tmvec_DB6_outputs_{THRESHOLD:.2f}")
COUNTS_FILE = f"protein_counts_per_genome_tmvec_{THRESHOLD:.2f}.tsv"
FUNCTIONS_FILE = f"sialic_functions_tmvec_{THRESHOLD:.2f}.tsv"

## 1. Check outputs

Marks a gene x proteome combination as `ok` in `tmvec_status.csv` when its output file exists in `RAW_DIR` and is not empty.
Expected file name: `{gene}_DB6_tmvec_database_{proteome}.tsv`.

In [8]:
status = pd.read_csv(STATUS_CSV)

def check_status(row):
    expected_file = RAW_DIR / f"{row['gene']}_DB6_tmvec_database_{row['proteome']}.tsv"
    if expected_file.exists() and expected_file.stat().st_size > 0:
        return "ok"
    return row["status"] if pd.notna(row["status"]) else ""

status["status"] = status.apply(check_status, axis=1)
status.to_csv(STATUS_CSV, index=False)

print(f"Completed: {(status['status'] == 'ok').sum()} of {len(status)}")

Completed: 727 of 837


## 2. Filter hits by TM-score

Keeps rows with TM-score >= `THRESHOLD`. Files with no remaining hits are not written.

In [9]:
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

n_files = 0
n_files_with_hits = 0
n_rows_total = 0

for tsv_file in RAW_DIR.glob("*.tsv"):
    n_files += 1
    hits = pd.read_csv(tsv_file, sep="\t")

    filtered = hits[hits["tm-score"] >= THRESHOLD]

    if not filtered.empty:
        filtered.to_csv(FILTERED_DIR / tsv_file.name, sep="\t", index=False)
        n_files_with_hits += 1
        n_rows_total += len(filtered)

print(f"Files processed: {n_files}")
print(f"Files with hits >= {THRESHOLD}: {n_files_with_hits}")
print(f"Total filtered rows: {n_rows_total}")

Files processed: 759
Files with hits >= 0.9: 321
Total filtered rows: 7891


## 3. Count table (proteome x gene)

For each filtered file, counts the number of **distinct query proteins** (`query_id`) with at least one hit, not the total number of rows.

In [10]:
pattern = re.compile(r"^(.+?)_DB6_tmvec_database_(.+)\.tsv$")

records = []

for tsv_file in FILTERED_DIR.glob("*.tsv"):
    match = pattern.match(tsv_file.name)
    if not match:
        print(f"[WARNING] Unexpected file name, skipping: {tsv_file.name}")
        continue

    gene, proteome = match.groups()

    hits = pd.read_csv(tsv_file, sep="\t")
    n_hits = hits["query_id"].nunique()   # distinct query proteins, not total rows

    records.append({"proteome": proteome, "gene": gene, "n_hits": n_hits})

long_df = pd.DataFrame(records)

matrix = long_df.pivot_table(
    index="proteome",
    columns="gene",
    values="n_hits",
    fill_value=0,
).astype(int)

matrix.to_csv(COUNTS_FILE, sep="\t")

print(f"Matrix: {matrix.shape[0]} proteomes x {matrix.shape[1]} genes")
print(f"Saved to: {COUNTS_FILE}")

Matrix: 23 proteomes x 33 genes
Saved to: protein_counts_per_genome_tmvec_0.90.tsv


## 4. Pathway presence/absence

A function is marked `OK` for a genome when:

| Function | Rule |
|---|---|
| Sialylation | any of *lst, cpsK, lic3A, lic3B, neuS* |
| Catabolism | all of *nanE, nanA, nanK* |
| Transport | any of *satA, satB, satC, satD, siaP, siaQ, siaM, siaT, nanU, nanT, nanC, ompC, ompF* |
| Sialidase | any of *nanH, SIAE* |
| Synthesis | both *neuB* and *neuC* |

In [11]:
counts = pd.read_csv(COUNTS_FILE, sep="\t", index_col=0)

def has_any(row, genes):
    return any(row.get(g, 0) > 0 for g in genes)

def has_all(row, genes):
    return all(row.get(g, 0) > 0 for g in genes)

results = []

for genome, row in counts.iterrows():
    res = {}

    if has_any(row, ["lst", "cpsK", "lic3A", "lic3B", "neuS"]):
        res["Sialylation"] = "OK"

    if has_all(row, ["nanE", "nanA", "nanK"]):
        res["Catabolism"] = "OK"

    if has_any(row, ["satA", "satB", "satC", "satD", "siaP", "siaQ", "siaM", "siaT",
                     "nanU", "nanT", "nanC", "ompC", "ompF"]):
        res["Transport"] = "OK"

    if has_any(row, ["nanH", "SIAE"]):
        res["Sialidase"] = "OK"

    if has_all(row, ["neuB", "neuC"]):
        res["Synthesis"] = "OK"

    res["genome"] = genome
    results.append(res)

functions = pd.DataFrame(results).set_index("genome").fillna("")
functions.to_csv(FUNCTIONS_FILE, sep="\t")

functions

,Transport,Sialidase,Catabolism,Sialylation,Synthesis
genome,,,,,
Bacter_fragilis,OK,OK,,,
Bifido_bifidum,OK,OK,,,
C_acnes,OK,OK,OK,,
C_difficile_630,OK,OK,OK,,
C_perfringens,OK,OK,OK,,
DH10B_ncbi,OK,,OK,,
E_coli_k1,OK,OK,OK,OK,OK
Fusobac,OK,,,,OK
GBS_515,OK,OK,OK,OK,OK
